# CBCT-1 — Сегментация костей/зубов (ToothFairy3 / nnU-Net)

Тонкий гайд: нарратив + вызовы скриптов. Тяжёлая логика — в `scripts/`.

**Почему этот эксперимент первый:** единственное направление с данными на руках
(ToothFairy3 публичный). Калибровка FEA ждёт пар до/после; ML мягких тканей ждёт
разметки кожи (её в ToothFairy3 нет). Кости/зубы — есть данные, метрика, запуск сегодня.

**Место в пайплайне:** хорошая костная сегментация → хороший меш → честная FEA.

---
## Гипотеза на бумаге (заполни ДО запуска)

- **Baseline прогноз:** nnU-Net ResEnc на 3d_fullres даст mean Dice ≈ ___ (литература ToothFairy2: ~0.90–0.92).
- **Абляция large_patch:** L40 (48 ГБ) против RTX 4090 (24 ГБ) → больше patch → больше контекста.
  Прогноз прироста Dice: +___ . Особенно жду улучшения на ___ (нумерация зубов? крупные кости?).
- **Контр-гипотеза:** прирост упрётся не в контекст, а в баланс классов / качество разметки.
  Тогда patch не поможет. Это тоже результат — он покажет реальное узкое место.

> Правило: предсказание записывается ДО запуска. Иначе это подгонка, а не проверка.


## 1. Окружение и smoke-test (на машине с L40)

In [ ]:
# Один раз: bash scripts/00_setup_env.sh  (из терминала, не из ноутбука)
# Здесь — только проверка, что переменные и данные на месте.
!python scripts/02_smoke_test.py --dataset 301

## 2. Данные
ToothFairy3 требует регистрации (ditto.ing.unimore.it). Скачай вручную, положи в `data/raw/`,
затем сконвертируй в формат nnU-Net. Отредактируй `labels` в dataset.json под нужные классы
(можно начать с подмножества: mandible/maxilla/teeth).

In [ ]:
# !python scripts/01_prepare_data.py --raw data/raw --out data/nnunet/raw --dataset-id 301
# !nnUNetv2_plan_and_preprocess -d 301 --verify_dataset_integrity
print("после препроцессинга — снова smoke-test, потом обучение")

## 3. Обучение (ночь на L40)
Baseline сначала, абляция — следующей ночью. Это и есть наполнение GPU,
пока днём ты идёшь по пяти глубинным экспериментам.

In [ ]:
# Ночь 1:
# !bash scripts/03_train.sh 301 baseline
# Ночь 2 (сначала сгенерируй план по configs/large_patch.md):
# !bash scripts/03_train.sh 301 large_patch
print("чекпоинты и логи nnU-Net пишет сам в nnUNet_results")

## 4. Оценка → артефакт
Артефакт = чекпоинт + metrics.json + таблица абляций. Без метрики на held-out это не артефакт.

In [ ]:
# !python scripts/04_evaluate.py \
#     --pred-dir data/predictions/baseline \
#     --gt-dir data/nnunet/raw/Dataset301_ToothFairy3/labelsTs \
#     --labels 1,2,3 --out metrics_baseline.json
print("повтори для large_patch -> metrics_largepatch.json")

## 5. Сверка предсказания с результатом (заполни ПОСЛЕ прогонов)

| Конфиг | patch_size | Предсказание Dice | Факт Dice | HD95 | Сошлось? |
|---|---|---|---|---|---|
| baseline | 160×320×320 | … | … | … | … |
| large_patch | 192×384×384 | … | … | … | … |

**Где теория сошлась / где ошиблась:**
_…твои заметки. Если large_patch не помог — какое узкое место это вскрывает?_

**Вывод для пайплайна:** какой чекпоинт идёт дальше в меш (Gmsh/TetGen)?

**Перенос навыка:** что из этого применимо к сегментации мягких тканей и калибровке материала,
когда появятся твои данные?
